# Random Forest Regressor (grid search + cross-validation)
Same split, preprocessing, CV folds and primary metric (RMSE) as notebooks 01 and 02. Tuning uses the training set only; the test set is not touched here.

Note: with only 5 features, `"sqrt"` and `"log2"` both give 2 features per split, so they would test the same setting twice.
The grid therefore uses `None` (all 5 features), `2` and `3` instead.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42


DATA_PATH = next(
    p for p in [Path("data/original/calorie_dataset.csv"),
                Path("../data/original/calorie_dataset.csv"),
                Path("calorie_dataset.csv")]
    if p.exists()
)
df = pd.read_csv(DATA_PATH)

FEATURES = ["protein_g", "carbs_g", "fat_g", "fiber_g", "sugar_g"]
TARGET = "calories"         

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)
print("Train:", X_train.shape, "| Test:", X_test.shape, "(test set is NOT used in this notebook)")

Train: (4000, 5) | Test: (1000, 5) (test set is NOT used in this notebook)


In [ ]:

preprocessor = ColumnTransformer(transformers=[("num", StandardScaler(), FEATURES)])
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

SCORING = {
    "rmse": "neg_root_mean_squared_error", 
    "mae":  "neg_mean_absolute_error",
    "r2":   "r2",
}

def summarize_cv(model, name):
    """5-fold CV on the TRAINING set only; prints mean +/- std for RMSE, MAE, R2."""
    res = cross_validate(model, X_train, y_train, cv=cv, scoring=SCORING)
    rmse, mae, r2 = -res["test_rmse"], -res["test_mae"], res["test_r2"]
    print(name)
    print(f"CV RMSE: {rmse.mean():.4f} +/- {rmse.std():.4f}")
    print(f"CV MAE : {mae.mean():.4f} +/- {mae.std():.4f}")
    print(f"CV R2  : {r2.mean():.4f} +/- {r2.std():.4f}")
    return {"model": name, "rmse_mean": rmse.mean(), "rmse_std": rmse.std(),
            "mae_mean": mae.mean(), "mae_std": mae.std(),
            "r2_mean": r2.mean(), "r2_std": r2.std()}

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV

rf_pipe = Pipeline([
    ("pre", preprocessor),
    ("model", RandomForestRegressor(random_state=RANDOM_STATE))
])

rf_param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [5, 10, None],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": [None, 2, 3],
}

In [4]:
rf_grid = GridSearchCV(
    rf_pipe,
    rf_param_grid,
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
)
rf_grid.fit(X_train, y_train)

print("Best params :", rf_grid.best_params_)
print("Best CV RMSE:", -rf_grid.best_score_)

Best params : {'model__max_depth': 10, 'model__max_features': 3, 'model__min_samples_leaf': 2, 'model__n_estimators': 200}
Best CV RMSE: 33.405492254226274


In [5]:
best_rf = rf_grid.best_estimator_
rf_results = summarize_cv(best_rf, "Random Forest")

Random Forest
CV RMSE: 33.4055 +/- 0.8526
CV MAE : 26.3299 +/- 0.5911
CV R2  : 0.9205 +/- 0.0037
